# Graph Theory — Lab Notebook (SOLUTIONS)
## DAT0222 — Albert School

Sessions 1-3: Networks, Traversals & Shortest Paths

**Instructor version — all exercises solved.**

---

## Setup

Run this cell first to install dependencies and import libraries.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install networkx matplotlib numpy

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from collections import deque
import heapq

---
# Part 1: Introduction to Networks (Session 1)
---

## 1.1 Getting Started with NetworkX

NetworkX is a Python library for creating, manipulating, and studying graphs.

In [ ]:
# Create an undirected graph
G = nx.Graph()
G.add_nodes_from(['A', 'B', 'C', 'D', 'E'])
G.add_edges_from([('A','B'), ('A','C'), ('B','D'), ('C','D'), ('D','E')])

# Visualize
nx.draw(G, with_labels=True, node_color='lightblue',
        node_size=700, font_size=16)
plt.title("Our first graph")
plt.show()

In [ ]:
# Explore basic properties
print("Nodes:", list(G.nodes()))
print("Edges:", list(G.edges()))
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

In [ ]:
# Degrees and neighbors
print("Degrees:", dict(G.degree()))
print("Neighbors of D:", list(G.neighbors('D')))

In [ ]:
# Adjacency matrix
A = nx.adjacency_matrix(G).todense()
print("Adjacency matrix:")
print(A)

In [ ]:
# Directed graph
D = nx.DiGraph()
D.add_edges_from([(1,2), (1,3), (2,3), (2,4), (3,2)])

print("In-degree:", dict(D.in_degree()))
print("Out-degree:", dict(D.out_degree()))

nx.draw(D, with_labels=True, node_color='lightgreen',
        node_size=700, font_size=16, arrows=True)
plt.title("Directed graph")
plt.show()

## 1.2 Weighted Graphs

In [ ]:
# Create a weighted graph
W = nx.Graph()
W.add_weighted_edges_from([
    ('A', 'B', 3),
    ('A', 'C', 1),
    ('B', 'C', 4),
    ('B', 'D', 2),
    ('C', 'D', 5)
])

# Draw with weights displayed
pos = nx.spring_layout(W, seed=42)
nx.draw(W, pos, with_labels=True, node_color='lightyellow',
        node_size=700, font_size=16)
edge_labels = nx.get_edge_attributes(W, 'weight')
nx.draw_networkx_edge_labels(W, pos, edge_labels=edge_labels, font_size=14)
plt.title("Weighted graph")
plt.show()

In [ ]:
# Access weight of a specific edge
print("Weight of edge A-B:", W['A']['B']['weight'])

## 1.3 Exercise 1 — Graph Vocabulary (SOLUTIONS)

**Graph 1 (undirected):** $V = \{1,2,3,4,5\}$, $E = \{(1,2),(1,3),(2,3),(3,4),(4,5),(2,5)\}$

**Graph 2 (directed):** $V = \{A,B,C,D\}$, $E = \{(A,B),(B,C),(C,A),(A,D)\}$

In [ ]:
# Build Graph 1 (undirected)
G1 = nx.Graph()
G1.add_edges_from([(1,2), (1,3), (2,3), (3,4), (4,5), (2,5)])

nx.draw(G1, with_labels=True, node_color='lightblue',
        node_size=700, font_size=16)
plt.title("Graph 1")
plt.show()

In [ ]:
# Build Graph 2 (directed)
G2 = nx.DiGraph()
G2.add_edges_from([('A','B'), ('B','C'), ('C','A'), ('A','D')])

nx.draw(G2, with_labels=True, node_color='lightgreen',
        node_size=700, font_size=16, arrows=True)
plt.title("Graph 2")
plt.show()

In [ ]:
# Question 1: Degree of each vertex in Graph 1
print("Degrees:", dict(G1.degree()))
# Expected: {1: 2, 2: 3, 3: 3, 4: 2, 5: 2}
# Sum = 2+3+3+2+2 = 12 = 2 * 6 edges ✓

In [ ]:
# Question 2: All paths from node 1 to node 5 in Graph 1
paths = list(nx.all_simple_paths(G1, 1, 5))
for p in paths:
    print(p)
# Expected: [1, 2, 5], [1, 2, 3, 4, 5], [1, 3, 2, 5], [1, 3, 4, 5]

In [ ]:
# Question 3: Cycles in Graph 1
print("Cycle basis:", nx.cycle_basis(G1))
# Expected: [[1, 2, 3], [2, 5, 4, 3]] (or similar)

In [ ]:
# Question 4: In-degree and out-degree for Graph 2
print("In-degrees:", dict(G2.in_degree()))
print("Out-degrees:", dict(G2.out_degree()))
# In-degrees: A=1, B=1, C=1, D=1
# Out-degrees: A=2, B=1, C=1, D=0
# D is a sink (out-degree 0). Cycle: A->B->C->A

In [ ]:
# Question 5: Adjacency matrix for Graph 1
print("Adjacency matrix:")
print(nx.adjacency_matrix(G1).todense())

---
# Part 2: Graph Traversals — BFS (Session 2)
---

## 2.1 BFS from Scratch

In [ ]:
def bfs(graph, start):
    """
    Breadth-First Search.
    
    Args:
        graph: dict {node: [neighbors]}
        start: starting node
    Returns:
        list of nodes in BFS visit order
    """
    visited = set()
    queue = deque([start])
    visited.add(start)
    order = []

    while queue:
        node = queue.popleft()        # dequeue from front
        order.append(node)

        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor) # enqueue at back

    return order

In [ ]:
# Test BFS
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'D'],
    'D': ['B', 'C'],
    'E': ['B']
}

print("BFS order:", bfs(graph, 'A'))

## 2.2 BFS with Distances

In [ ]:
def bfs_distances(graph, start):
    """
    BFS that also computes shortest distances and parent pointers.
    
    Returns:
        distances: dict {node: distance from start}
        parent: dict {node: parent node in BFS tree}
    """
    visited = set([start])
    queue = deque([start])
    distances = {start: 0}
    parent = {start: None}

    while queue:
        node = queue.popleft()
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)
                distances[neighbor] = distances[node] + 1
                parent[neighbor] = node

    return distances, parent

In [ ]:
dist, parent = bfs_distances(graph, 'A')
print("Distances:", dist)
print("Parents:", parent)

## 2.3 BFS with NetworkX

In [ ]:
G = nx.Graph()
G.add_edges_from([('A','B'), ('A','C'), ('B','D'), ('B','E'), ('C','D')])

# BFS traversal edges
print("BFS edges:", list(nx.bfs_edges(G, 'A')))

# BFS tree
T = nx.bfs_tree(G, 'A')
print("BFS tree edges:", list(T.edges()))

# Shortest path (unweighted = BFS)
print("Shortest path A->E:", nx.shortest_path(G, 'A', 'E'))
print("Shortest path length:", nx.shortest_path_length(G, 'A', 'E'))

# All shortest path lengths from A
print("All distances from A:", dict(nx.shortest_path_length(G, 'A')))

## 2.4 Exercise 2 — Implement BFS (SOLUTIONS)

In [ ]:
# Graph for Exercise 2
exercise_graph = {
    1: [2, 3],
    2: [1, 4, 5],
    3: [1, 5, 6],
    4: [2],
    5: [2, 3, 7],
    6: [3],
    7: [5]
}

In [ ]:
# Task 1: BFS from node 1
print("BFS order:", bfs(exercise_graph, 1))
# Expected: [1, 2, 3, 4, 5, 6, 7]
# Level 0: 1 | Level 1: 2, 3 | Level 2: 4, 5, 6 | Level 3: 7

In [ ]:
# Task 2: BFS with distances
dist, parent = bfs_distances(exercise_graph, 1)
print("Distances:", dist)
print("Distance from 1 to 7:", dist[7])  # Expected: 3

In [ ]:
# Task 3: Verify with NetworkX
G_ex = nx.Graph()
for node, neighbors in exercise_graph.items():
    for neighbor in neighbors:
        G_ex.add_edge(node, neighbor)

print("BFS edges:", list(nx.bfs_edges(G_ex, 1)))
print("Shortest path 1->7:", nx.shortest_path(G_ex, 1, 7))
print("Distance 1->7:", nx.shortest_path_length(G_ex, 1, 7))

## 2.5 Connectivity

In [ ]:
# Check connectivity from scratch using BFS
def is_connected(graph):
    """Check if an undirected graph is connected using BFS."""
    start = next(iter(graph))
    visited = bfs(graph, start)
    return len(visited) == len(graph)

def connected_components(graph):
    """Find all connected components using BFS."""
    visited = set()
    components = []
    for node in graph:
        if node not in visited:
            component = bfs(graph, node)
            visited.update(component)
            components.append(component)
    return components

In [ ]:
# Test on a disconnected graph
disconnected = {
    1: [2, 3],
    2: [1, 3],
    3: [1, 2],
    4: [5],
    5: [4]
}

print("Connected?", is_connected(disconnected))
print("Components:", connected_components(disconnected))

In [ ]:
# Connectivity with NetworkX
G = nx.Graph()
G.add_edges_from([(1,2), (2,3), (1,3), (4,5)])

print("Connected?", nx.is_connected(G))
print("Components:", list(nx.connected_components(G)))
print("Number of components:", nx.number_connected_components(G))

In [ ]:
# Strong connectivity for directed graphs
D = nx.DiGraph([(1,2), (2,3), (3,1), (1,4)])

print("Strongly connected?", nx.is_strongly_connected(D))
print("SCCs:", list(nx.strongly_connected_components(D)))
print("Weakly connected?", nx.is_weakly_connected(D))

---
# Part 3: Graph Traversals — DFS (Session 2 cont.)
---

## 3.1 DFS Iterative

In [ ]:
def dfs(graph, start):
    """
    Depth-First Search using an explicit stack.
    """
    visited = set()
    stack = [start]
    order = []

    while stack:
        node = stack.pop()              # pop from top (LIFO)
        if node not in visited:
            visited.add(node)
            order.append(node)
            for neighbor in reversed(graph[node]):
                if neighbor not in visited:
                    stack.append(neighbor)

    return order

In [ ]:
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'D'],
    'D': ['B', 'C'],
    'E': ['B']
}

print("DFS order:", dfs(graph, 'A'))
print("BFS order:", bfs(graph, 'A'))

## 3.2 DFS Recursive

In [ ]:
def dfs_recursive(graph, node, visited=None):
    """
    DFS using recursion (the call stack IS the stack).
    """
    if visited is None:
        visited = set()

    visited.add(node)
    print(node, end=' ')

    for neighbor in graph[node]:
        if neighbor not in visited:
            dfs_recursive(graph, neighbor, visited)

    return visited

In [ ]:
print("DFS recursive order:")
dfs_recursive(graph, 'A')
print()

## 3.3 DFS with NetworkX

In [ ]:
G = nx.Graph()
G.add_edges_from([('A','B'), ('A','C'), ('B','D'), ('B','E'), ('C','D')])

print("DFS edges:", list(nx.dfs_edges(G, 'A')))
T = nx.dfs_tree(G, 'A')
print("DFS tree edges:", list(T.edges()))
print("DFS preorder:", list(nx.dfs_preorder_nodes(G, 'A')))
print("DFS postorder:", list(nx.dfs_postorder_nodes(G, 'A')))

## 3.4 Exercise 3 — DFS + Compare with BFS (SOLUTIONS)

In [ ]:
# Graph for Exercise 3
exercise_graph = {
    1: [2, 3],
    2: [1, 4, 5],
    3: [1, 5, 6],
    4: [2],
    5: [2, 3, 7],
    6: [3],
    7: [5]
}

In [ ]:
# Task 1: DFS (iterative) from node 1
print("DFS order:", dfs(exercise_graph, 1))
# Expected: [1, 2, 4, 5, 3, 6, 7] (goes deep first)

In [ ]:
# Task 2: DFS (recursive) from node 1
print("DFS recursive:")
dfs_recursive(exercise_graph, 1)
print()

In [ ]:
# Task 3: Compare BFS vs DFS
print("BFS:", bfs(exercise_graph, 1))
print("DFS:", dfs(exercise_graph, 1))
# BFS: level-by-level [1, 2, 3, 4, 5, 6, 7]
# DFS: deep-first    [1, 2, 4, 5, 3, 6, 7]
# Different orders! Same set of nodes visited.

In [ ]:
# Task 4: Graph with 3 connected components
multi_comp = {
    1: [2, 3],
    2: [1, 3],
    3: [1, 2],
    4: [5],
    5: [4],
    6: []
}

print("Components (from scratch):", connected_components(multi_comp))

# Verify with NetworkX
G_mc = nx.Graph()
G_mc.add_edges_from([(1,2), (2,3), (1,3), (4,5)])
G_mc.add_node(6)
print("Components (NetworkX):", list(nx.connected_components(G_mc)))

In [ ]:
# Task 5: Directed graph with SCCs
D = nx.DiGraph()
D.add_edges_from([(1,2), (2,3), (3,1), (1,4)])

print("Strongly connected?", nx.is_strongly_connected(D))
print("SCCs:", list(nx.strongly_connected_components(D)))
# Expected: SCCs are {1,2,3} and {4}

---
# Part 4: Shortest Paths (Session 3)
---

## 4.1 Dijkstra from Scratch

In [ ]:
def dijkstra(graph, start):
    """
    Dijkstra's shortest path algorithm.
    
    Args:
        graph: dict {node: [(neighbor, weight), ...]}
        start: source node
    Returns:
        distances: dict {node: shortest distance from start}
        predecessors: dict {node: previous node on shortest path}
    """
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    predecessors = {start: None}
    pq = [(0, start)]  # (distance, node)

    while pq:
        current_dist, u = heapq.heappop(pq)

        if current_dist > distances[u]:
            continue  # skip outdated entry

        for v, weight in graph[u]:
            new_dist = current_dist + weight
            if new_dist < distances[v]:          # relaxation
                distances[v] = new_dist
                predecessors[v] = u
                heapq.heappush(pq, (new_dist, v))

    return distances, predecessors

In [ ]:
# Test Dijkstra
weighted_graph = {
    'A': [('B', 7), ('E', 1)],
    'B': [('A', 7), ('C', 3), ('E', 8)],
    'C': [('B', 3), ('D', 6)],
    'D': [('C', 6), ('E', 7)],
    'E': [('A', 1), ('B', 8), ('D', 7)]
}

dist, pred = dijkstra(weighted_graph, 'A')
print("Distances from A:", dist)
print("Predecessors:", pred)

## 4.2 Reconstructing the Path

In [ ]:
def shortest_path(graph, start, end):
    """Find shortest path and its weight using Dijkstra."""
    distances, predecessors = dijkstra(graph, start)

    # Reconstruct path by following predecessors backwards
    path = []
    node = end
    while node is not None:
        path.append(node)
        node = predecessors.get(node)
    path.reverse()

    return path, distances[end]

In [ ]:
path, dist = shortest_path(weighted_graph, 'A', 'C')
print(f"Shortest path A->C: {path}, distance: {dist}")

## 4.3 Dijkstra with NetworkX

In [ ]:
W = nx.Graph()
W.add_weighted_edges_from([
    ('A','B',7), ('A','E',1), ('B','C',3),
    ('B','E',8), ('C','D',6), ('D','E',7)
])

print("Path A->C:", nx.dijkstra_path(W, 'A', 'C'))
print("Distance:", nx.dijkstra_path_length(W, 'A', 'C'))
print("All distances from A:", dict(nx.single_source_dijkstra_path_length(W, 'A')))

In [ ]:
# Visualize with shortest path highlighted
path = nx.dijkstra_path(W, 'A', 'C')
path_edges = list(zip(path, path[1:]))
edge_colors = ['red' if e in path_edges or (e[1],e[0]) in path_edges
               else 'gray' for e in W.edges()]
pos = nx.spring_layout(W, seed=42)
nx.draw(W, pos, with_labels=True, edge_color=edge_colors, width=2,
        node_color='lightyellow', node_size=700)
nx.draw_networkx_edge_labels(W, pos,
        edge_labels=nx.get_edge_attributes(W, 'weight'))
plt.title("Shortest path A→C highlighted in red")
plt.show()

## 4.4 Exercise 4 — Verify Dijkstra by Hand (SOLUTIONS)

```
    S --4-- A --2-- D
    |       |       |
    2       1       3
    |       |       |
    B --3-- C --5-- E
```

Edges: S-A:4, S-B:2, A-C:1, A-D:2, B-C:3, C-E:5, D-E:3

In [ ]:
# Graph for Exercise 4
ex4_graph = {
    'S': [('A', 4), ('B', 2)],
    'A': [('S', 4), ('C', 1), ('D', 2)],
    'B': [('S', 2), ('C', 3)],
    'C': [('A', 1), ('B', 3), ('E', 5)],
    'D': [('A', 2), ('E', 3)],
    'E': [('C', 5), ('D', 3)]
}

In [ ]:
# Verify distances
dist, pred = dijkstra(ex4_graph, 'S')
print("Distances from S:", dist)
# Expected: S=0, B=2, A=4, C=5, D=6, E=9

# Iteration table:
# Step 1: Process S (d=0). Relax S-A: d[A]=4, S-B: d[B]=2.
# Step 2: Process B (d=2). Relax B-C: d[C]=2+3=5.
# Step 3: Process A (d=4). Relax A-C: 4+1=5 (no change). A-D: d[D]=4+2=6.
# Step 4: Process C (d=5). Relax C-E: d[E]=5+5=10.
# Step 5: Process D (d=6). Relax D-E: 6+3=9 < 10, d[E]=9! Updated!
# Step 6: Process E (d=9). Done.

In [ ]:
# Shortest path from S to E
path, distance = shortest_path(ex4_graph, 'S', 'E')
print(f"Path: {path}, Distance: {distance}")
# Expected: Path: ['S', 'A', 'D', 'E'], Distance: 9

## 4.5 Exercise 5 — Implement Dijkstra (SOLUTIONS)

In [ ]:
def my_dijkstra(graph, start):
    """
    Student implementation of Dijkstra's algorithm.
    """
    distances = {node: float('inf') for node in graph}
    distances[start] = 0
    predecessors = {start: None}
    pq = [(0, start)]

    while pq:
        current_dist, u = heapq.heappop(pq)

        if current_dist > distances[u]:
            continue

        for v, weight in graph[u]:
            new_dist = current_dist + weight
            if new_dist < distances[v]:
                distances[v] = new_dist
                predecessors[v] = u
                heapq.heappush(pq, (new_dist, v))

    return distances, predecessors

In [ ]:
# Test on Exercise 4 graph
dist, pred = my_dijkstra(ex4_graph, 'S')
print("Distances:", dist)
print("Predecessors:", pred)

In [ ]:
# Verify with NetworkX
W_ex4 = nx.Graph()
W_ex4.add_weighted_edges_from([
    ('S','A',4), ('S','B',2), ('A','C',1),
    ('A','D',2), ('B','C',3), ('C','E',5), ('D','E',3)
])

print("NX path S->E:", nx.dijkstra_path(W_ex4, 'S', 'E'))
print("NX distance:", nx.dijkstra_path_length(W_ex4, 'S', 'E'))

In [ ]:
# Visualize with shortest path highlighted
path = nx.dijkstra_path(W_ex4, 'S', 'E')
path_edges = list(zip(path, path[1:]))
edge_colors = ['red' if e in path_edges or (e[1],e[0]) in path_edges
               else 'gray' for e in W_ex4.edges()]
pos = nx.spring_layout(W_ex4, seed=42)
nx.draw(W_ex4, pos, with_labels=True, edge_color=edge_colors, width=2,
        node_color='lightyellow', node_size=700)
nx.draw_networkx_edge_labels(W_ex4, pos,
        edge_labels=nx.get_edge_attributes(W_ex4, 'weight'))
plt.title("Shortest path S→E highlighted in red")
plt.show()

In [ ]:
# Explore: Add edge B-E with weight 1
W_ex4_mod = W_ex4.copy()
W_ex4_mod.add_weighted_edges_from([('B', 'E', 1)])

print("New path S->E:", nx.dijkstra_path(W_ex4_mod, 'S', 'E'))
print("New distance:", nx.dijkstra_path_length(W_ex4_mod, 'S', 'E'))
# Expected: S->B->E with distance 2+1=3 (much shorter than 9!)

## 4.6 Bellman-Ford (brief)

In [ ]:
# Bellman-Ford with NetworkX
W = nx.Graph()
W.add_weighted_edges_from([
    ('S','A',4), ('S','B',2), ('A','C',1),
    ('A','D',2), ('B','C',3), ('C','E',5), ('D','E',3)
])

print("Bellman-Ford path S->E:", nx.bellman_ford_path(W, 'S', 'E'))
print("Bellman-Ford distance:", nx.bellman_ford_path_length(W, 'S', 'E'))

## 4.7 Exercise 6 — Paris Metro Challenge (SOLUTIONS)

In [ ]:
# Metro graph
metro = nx.Graph()
metro.add_weighted_edges_from([
    ('Châtelet', 'Gare du Nord', 5),
    ('Châtelet', 'Bastille', 4),
    ('Gare du Nord', 'République', 3),
    ('République', 'Bastille', 4),
    ('République', 'Nation', 7),
    ('Bastille', 'Nation', 3),
    ('Nation', 'Gare de Lyon', 5),
    ('Châtelet', 'Gare de Lyon', 8),
])

# Visualize
pos = nx.spring_layout(metro, seed=42)
nx.draw(metro, pos, with_labels=True, node_color='lightyellow',
        node_size=1000, font_size=10)
nx.draw_networkx_edge_labels(metro, pos,
        edge_labels=nx.get_edge_attributes(metro, 'weight'))
plt.title("Paris Metro (simplified)")
plt.show()

In [ ]:
# Q1: Fastest route from Châtelet to Nation
path = nx.dijkstra_path(metro, 'Châtelet', 'Nation')
length = nx.dijkstra_path_length(metro, 'Châtelet', 'Nation')
print(f"Fastest route: {path}")
print(f"Travel time: {length} minutes")
# Expected: Châtelet -> Bastille -> Nation = 4+3 = 7 minutes

In [ ]:
# Q2: Station with most connections (highest degree)
degrees = dict(metro.degree())
print("Degrees:", degrees)
max_station = max(degrees, key=degrees.get)
print(f"Most connected: {max_station} (degree {degrees[max_station]})")
# Châtelet and République both have degree 3

In [ ]:
# Q3: Remove République — can we still get from Châtelet to Nation?
metro_no_rep = metro.copy()
metro_no_rep.remove_node('République')

print("Still connected?", nx.has_path(metro_no_rep, 'Châtelet', 'Nation'))
path = nx.dijkstra_path(metro_no_rep, 'Châtelet', 'Nation')
length = nx.dijkstra_path_length(metro_no_rep, 'Châtelet', 'Nation')
print(f"New fastest route: {path}")
print(f"New travel time: {length} minutes")
# Expected: Châtelet -> Bastille -> Nation = 7 min (same as before!)